In [1]:
from utils.std_model import base_model

chatLLM = base_model()
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_extract = ChatPromptTemplate.from_template("从以下文本中提取技术规格：\n\n{text_input}")
prompt_transform = ChatPromptTemplate.from_template(
    "将以下规格转换为 JSON 对象，使用 'cpu'、'memory' 和 'storage' 作为键：\n\n{specifications}"
)
extraction_chain = prompt_extract | chatLLM | StrOutputParser()

full_chain = (
    {"specifications": extraction_chain}
    | prompt_transform
    | chatLLM
    | StrOutputParser()
)
input_text = "新款笔记本电脑型号配备 3.5 GHz 八核处理器、16GB 内存和 1TB NVMe 固态硬盘。"

final_result = full_chain.invoke({"text_input": input_text})

final_result

'{\n  "cpu": "3.5 GHz 八核",\n  "memory": "16GB",\n  "storage": "1TB NVMe"\n}'

In [2]:
from typing import TypedDict

from langgraph.constants import END, START
from langgraph.graph import StateGraph

from utils.std_model import base_model

llm = base_model()


class JokeState(TypedDict):
    topic: str
    joke: str
    improved_joke: str


class HasPunchline(TypedDict):
    has_punchline: bool


def generate_joke(state: JokeState):
    """ 根据话题生成笑话 """
    topic = state["topic"]

    msg = llm.invoke(f"根据主题“{topic}”生成一个笑话")
    return {
        "joke": msg.content,
    }


def improve_joke(state: JokeState):
    """ 提升笑话的好笑程度 """

    msg = llm.invoke(f"通过加入文字游戏让这个笑话更好笑: {state['joke']}")
    return {"improved_joke": msg.content}


def check_joke(state: JokeState):
    joke = state['joke']

    llm_with_tools = llm.with_structured_output(HasPunchline)

    resp = llm_with_tools.invoke(f"判断笑话:[{joke}]是否有包袱？")

    if resp["has_punchline"]:
        return END
    else:
        return improve_joke.__name__


graph_builder = StateGraph(JokeState)

graph_builder.add_node(generate_joke.__name__, generate_joke)
graph_builder.add_node(improve_joke.__name__, improve_joke)

graph_builder.add_edge(START, generate_joke.__name__)
graph_builder.add_conditional_edges(generate_joke.__name__, check_joke, [END, improve_joke.__name__])
graph_builder.add_edge(improve_joke.__name__, END)

graph = graph_builder.compile()

init_state = {
    "topic": "猫",
}
graph.invoke(init_state)



{'topic': '猫',
 'joke': '一只猫对另一只猫说：“你知道吗？人类总以为我们听不懂他们的话。”  \n另一只猫回答：“对啊，其实我们不仅听得懂，还能用猫语回复，只是懒得理他们。”  \n第一只猫问：“那为什么我们有时会喵喵叫？”  \n第二只猫说：“那是我们在吐槽他们——比如刚才那个人类说‘猫咪真可爱’，我回了一句‘废话，还用你说’。”'}